# SASV: ECAPA + AASIST locked **eval** (report once)

Official SASV **eval** with AASIST CM:

```text
s_sasv = s_asv + P_bona(AASIST)
```

**Rules**

- Finish `09_ecapa_plus_aasist_sasv.ipynb` on full **dev** first.
- Do **not** change fusion after you see these eval numbers.
- Reuses ECAPA `s_asv` from `runs/ecapa_plus_lfcc_eval/scores_eval.csv` (notebook `04`).

Compare published B1-v2 (ECAPA+AASIST) ~**1.71%** SASV-EER on eval.

Needs local `aasist/` clone + `AASIST.pth`.

In [1]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd()
if not (ROOT / "aasist_fusion_lib.py").exists():
    ROOT = Path(r"D:\speaker-verification-system\replay-cnn-baseline\experiments\sasv_la2019")
sys.path.insert(0, str(ROOT))

import torch
from experiment_lib import DEFAULT_LA, DEFAULT_SASV, RUNS_DIR
from aasist_fusion_lib import DEFAULT_AASIST, score_ecapa_aasist_fusion

print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
print("LA:", DEFAULT_LA.exists())
print("SASV:", DEFAULT_SASV.exists())
print("AASIST:", DEFAULT_AASIST.exists())
print("weights:", (DEFAULT_AASIST / "models" / "weights" / "AASIST.pth").exists())

cuda: True
NVIDIA GeForce RTX 4060 Laptop GPU
LA: True
SASV: True
AASIST: True
weights: True


## Locked settings

- `SPLIT` fixed to **`eval`**
- `MAX_TRIALS = 0` → all eval trials
- Requires `runs/ecapa_plus_lfcc_eval/scores_eval.csv`

In [2]:
SPLIT = "eval"
MAX_TRIALS = 0
DEVICE = "cuda"
FORCE_CPU = False

ECAPA_CSV = RUNS_DIR / "ecapa_plus_lfcc_eval" / "scores_eval.csv"
assert SPLIT == "eval", "This notebook is for locked eval only"
print("ECAPA CSV:", ECAPA_CSV.exists(), ECAPA_CSV)
assert ECAPA_CSV.exists(), "Run notebook 04 first to create ecapa_plus_lfcc_eval scores"

ECAPA CSV: True D:\speaker-verification-system\replay-cnn-baseline\experiments\sasv_la2019\runs\ecapa_plus_lfcc_eval\scores_eval.csv


## Run ECAPA + AASIST (eval)

Writes `runs/ecapa_plus_aasist_eval/`

In [3]:
summary = score_ecapa_aasist_fusion(
    split=SPLIT,
    max_trials=MAX_TRIALS,
    device=DEVICE,
    force_cpu=FORCE_CPU,
    la_root=DEFAULT_LA,
    sasv_root=DEFAULT_SASV,
    aasist_root=DEFAULT_AASIST,
    ecapa_csv=ECAPA_CSV,
    output_dir=RUNS_DIR / f"ecapa_plus_aasist_{SPLIT}",
)
display({
    "system": summary["system"],
    "split": summary["split"],
    "n": summary["num_scored"],
    "unique_utts": summary["num_unique_test_utts"],
    "sasv_eer_%": summary["sasv_eer_percent"],
    "sv_eer_%": summary["sv_eer_percent"],
    "spf_eer_%": summary["spf_eer_percent"],
})

Trials: {'target': 5370, 'nontarget': 33327, 'spoof': 63882, 'total': 102579} | reuse s_asv from scores_eval.csv
AASIST on cuda


AASIST utts:   0%|          | 0/71237 [00:00<?, ?it/s]

{
  "system": "ecapa_plus_aasist_sum",
  "split": "eval",
  "max_trials": 0,
  "num_scored": 102579,
  "key_counts": {
    "target": 5370,
    "nontarget": 33327,
    "spoof": 63882,
    "total": 102579
  },
  "device": "cuda",
  "cm_backend": "aasist",
  "fusion": "s_asv + P_bona(AASIST softmax)",
  "ecapa_csv": "D:\\speaker-verification-system\\replay-cnn-baseline\\experiments\\sasv_la2019\\runs\\ecapa_plus_lfcc_eval\\scores_eval.csv",
  "num_unique_test_utts": 71237,
  "sasv_eer": 0.011359404096869757,
  "sv_eer": 0.008193668528864067,
  "spf_eer": 0.013884975424268716,
  "sasv_eer_percent": 1.1359404096869756,
  "sv_eer_percent": 0.8193668528864066,
  "spf_eer_percent": 1.3884975424268717
}


{'system': 'ecapa_plus_aasist_sum',
 'split': 'eval',
 'n': 102579,
 'unique_utts': 71237,
 'sasv_eer_%': 1.1359404096869756,
 'sv_eer_%': 0.8193668528864066,
 'spf_eer_%': 1.3884975424268717}

## Report table (eval + locked dev)

In [4]:
print("Eval:")
for label, path in [
    ("ecapa_only", RUNS_DIR / "ecapa_only_eval" / "metrics_eval.json"),
    ("ecapa_plus_lfcc", RUNS_DIR / "ecapa_plus_lfcc_eval" / "metrics_eval.json"),
    ("ecapa_plus_wavlm", RUNS_DIR / "ecapa_plus_wavlm_eval" / "metrics_eval.json"),
    ("ecapa_plus_aasist", RUNS_DIR / "ecapa_plus_aasist_eval" / "metrics_eval.json"),
]:
    if not path.exists():
        print("  Missing:", path)
        continue
    m = json.loads(path.read_text(encoding="utf-8"))
    print(
        f"  {label:20s}  SASV={m['sasv_eer_percent']:.4f}%  "
        f"SV={m['sv_eer_percent']:.4f}%  SPF={m['spf_eer_percent']:.4f}%  n={m.get('num_scored', '?')}"
    )

print("\nDev (reference):")
for label, path in [
    ("ecapa_only", RUNS_DIR / "ecapa_only_dev" / "metrics_dev.json"),
    ("ecapa_plus_lfcc", RUNS_DIR / "ecapa_plus_lfcc_dev" / "metrics_dev.json"),
    ("ecapa_plus_wavlm", RUNS_DIR / "ecapa_plus_wavlm_dev" / "metrics_dev.json"),
    ("ecapa_plus_aasist", RUNS_DIR / "ecapa_plus_aasist_dev" / "metrics_dev.json"),
]:
    if not path.exists():
        print("  Missing:", path)
        continue
    m = json.loads(path.read_text(encoding="utf-8"))
    print(
        f"  {label:20s}  SASV={m['sasv_eer_percent']:.4f}%  "
        f"SV={m['sv_eer_percent']:.4f}%  SPF={m['spf_eer_percent']:.4f}%"
    )

Eval:
  ecapa_only            SASV=20.6704%  SV=0.7635%  SPF=27.0452%  n=102579
  ecapa_plus_lfcc       SASV=7.1279%  SV=1.5642%  SPF=9.7070%  n=102579
  ecapa_plus_wavlm      SASV=12.2533%  SV=14.6927%  SPF=6.5363%  n=102579
  ecapa_plus_aasist     SASV=1.1359%  SV=0.8194%  SPF=1.3885%  n=102579

Dev (reference):
  ecapa_only            SASV=15.2291%  SV=1.2483%  SPF=17.9090%
  ecapa_plus_lfcc       SASV=1.1438%  SV=2.0978%  SPF=0.0897%
  ecapa_plus_wavlm      SASV=7.3450%  SV=11.8598%  SPF=3.8410%
  ecapa_plus_aasist     SASV=0.7412%  SV=1.3477%  SPF=0.1301%


## Done

Copy eval EERs into your table / `results.md`. Do not re-tune on these numbers.

Reference (published eval): B1-v2 (ECAPA+AASIST) ~**1.71%** SASV-EER.